# Phase 3.10 — Full BBA Training with GraphTransformerScoreModel

Clones `grapentt/riemannian-scoremd` from GitHub, loads precomputed data from Drive,
trains the graph transformer on all 62k BBA frames, and saves checkpoints + results back to Drive.

**Before running**: select a GPU runtime — Runtime → Change runtime type → A100 or T4.

**Expected**: ~11 min on A100/H100 at 0.22 ms/step. Loss drops from ~22 → <5 by epoch 3000.

## 0. GPU check + install deps

In [ ]:
import subprocess, sys

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU  :", result.stdout.strip() or "NOT FOUND — change runtime type!")

import jax
print("JAX  :", jax.__version__)
print("Devs :", jax.devices())

In [ ]:
# Colab ships JAX+CUDA; just add the extra deps we need.
# einops and orbax-checkpoint are the only ones not pre-installed.
!pip install -q einops orbax-checkpoint

## 1. Clone repo from GitHub

In [ ]:
import os, subprocess

REPO_DIR = "/content/riemannian-scoremd"
GITHUB_URL = "https://github.com/grapentt/riemannian-scoremd.git"

if os.path.isdir(REPO_DIR):
    print("Repo already cloned — pulling latest changes.")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    print(f"Cloning {GITHUB_URL} ...")
    subprocess.run(["git", "clone", GITHUB_URL, REPO_DIR], check=True)

# Verify key source files are present
for rel in [
    "src/manifold/pointcloud_jax.py",
    "src/diffusion/manifold_sde.py",
    "src/models/graph_transformer_jax.py",
    "src/training/train_manifold.py",
]:
    path = os.path.join(REPO_DIR, rel)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {status}  {rel}")

# Add src/ to Python path
SRC = os.path.join(REPO_DIR, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

print("\nRepo ready.")

## 2. Mount Drive and configure paths

**Edit `DRIVE_DATA_DIR` and `DRIVE_OUT_DIR`** to match where you stored the
precomputed BBA data on your Drive (`noised_r00.npz` … `noised_r09.npz`).
Results (checkpoints, loss curve, eval metrics) will be written to `DRIVE_OUT_DIR`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── EDIT THESE ─────────────────────────────────────────────────────────────
# Folder on Drive that contains noised_r00.npz … noised_r09.npz
DRIVE_DATA_DIR = "/content/drive/MyDrive/thesis-data/precomputed/bba"

# Where to write checkpoints and results on Drive
DRIVE_OUT_DIR  = "/content/drive/MyDrive/thesis-data/runs/bba_gt_phase310"
# ───────────────────────────────────────────────────────────────────────────

# Local scratch dirs (fast SSD on Colab VM)
LOCAL_DATA_DIR = "/content/data/precomputed/bba"
LOCAL_CKPT_DIR = "/content/checkpoints/bba_gt_phase310"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUT_DIR,  exist_ok=True)

# Verify precomputed files exist on Drive
import glob
drive_files = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, "noised_r*.npz")))
print(f"Found {len(drive_files)} precomputed files on Drive:")
for f in drive_files:
    print(f"  {os.path.basename(f)}")
assert len(drive_files) > 0, f"No noised_r*.npz files found in {DRIVE_DATA_DIR}"

## 2b. (Optional) Regenerate precomputed data — SKIP if data is on Drive

**✅ Data already on Drive (`thesis-data/precomputed/bba/noised_r00…r09.npz`)
— skip both cells in this section and continue at step 3.**

Only run this if Drive data is missing or stale (e.g. after changing `alpha` or
the `s_log` implementation). Each repeat takes ~8 min on Colab CPU; 10 repeats
≈ 80 min. The cell is a no-op if all 10 files already exist on Drive.
Results are pushed back to `DRIVE_DATA_DIR` automatically.

In [ ]:
# ── (Optional) Regenerate precomputed BBA data ──────────────────────────────
# NO-OP if all 10 files already exist on Drive.
# Only regenerate if Drive data is missing or the s_exp/s_log implementation changed.
#
# alpha=1.0 is correct for BBA and AK. s_log targets (Riemannian gradient).
# Expected time if regenerating: ~8 min/repeat × 10 repeats ≈ 80 min (CPU).

import subprocess, os, glob, shutil

N_REPEATS = 10

# Check how many files are already on Drive
existing = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, "noised_r*.npz")))
if len(existing) == N_REPEATS:
    print(f"✅ All {N_REPEATS} precomputed files already on Drive — nothing to do.")
    for f in existing:
        print(f"  {os.path.basename(f)}")
else:
    missing = N_REPEATS - len(existing)
    print(f"⚠️  {missing} files missing on Drive — regenerating all {N_REPEATS} repeats.")

    cmd = [
        "python", os.path.join(REPO_DIR, "scripts", "precompute_noised_data.py"),
        "--protein", "bba",
        "--n-repeats", str(N_REPEATS),
        "--output", "/content/data/precomputed",  # script appends /bba
        "--seed", "0",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    gen_files = sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "noised_r*.npz")))
    assert len(gen_files) == N_REPEATS, f"Expected {N_REPEATS}, got {len(gen_files)}"

    # Push to Drive
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for f in gen_files:
        dst = os.path.join(DRIVE_DATA_DIR, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f"  → Drive: {os.path.basename(f)}")
    print(f"\n✅ All {N_REPEATS} files saved to {DRIVE_DATA_DIR}")
    print("Download locally via Drive desktop sync or rclone.")


In [ ]:
# Copy precomputed files to local SSD for fast access during training.
# Drive I/O is slow (~50 MB/s); this copy takes ~30s and makes training 3-5× faster.
import shutil, time

t0 = time.time()
for src_f in drive_files:
    dst_f = os.path.join(LOCAL_DATA_DIR, os.path.basename(src_f))
    if not os.path.exists(dst_f):
        shutil.copy2(src_f, dst_f)
        print(f"  Copied {os.path.basename(src_f)}")
    else:
        print(f"  Already present: {os.path.basename(src_f)}")

local_files = sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "noised_r*.npz")))
print(f"\nLocal SSD: {len(local_files)} files  ({time.time()-t0:.0f}s)")

## 3. Imports and sanity checks

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from manifold.pointcloud_jax import ShapeManifold
from diffusion.manifold_sde import ManifoldVP
from models.graph_transformer_jax import GraphTransformerScoreModel
from training.train_manifold import train_from_precomputed

print("All imports OK")

# Quick data sanity check
sample = np.load(local_files[0])
print(f"\nData shape (r00):")
print(f"  x_t    : {sample['x_t'].shape}  {sample['x_t'].dtype}")
print(f"  s_true : {sample['s_true'].shape}")
print(f"  t      : {sample['t'].shape}  range [{sample['t'].min():.2f}, {sample['t'].max():.2f}]")
nan_xt = np.isnan(sample['x_t']).sum()
nan_st = np.isnan(sample['s_true']).sum()
print(f"  NaN x_t: {nan_xt}   NaN s_true: {nan_st}")

## 4. Configuration

In [ ]:
# ── Manifold ─────────────────────────────────────────────────────────────────
# BBA: 28 Cα atoms, d=3, manifold dim = 3*28 - 6 = 78
N_ATOMS = 28
D       = 3
ALPHA   = 1.0     # w^delta gyration weight (both BBA and AK use 1.0; ShapeManifold default is 0.1 but never used for proteins)

# ── Training ─────────────────────────────────────────────────────────────────
N_EPOCHS   = 3000
BATCH_SIZE = 64
LR         = 1e-3
GRAD_CLIP  = 1.0
EMA_DECAY  = 0.995
LOG_EVERY  = 50
SEED       = 42

# eps_parameterization: predict v_h_unit (unit-g-norm noise direction)
# instead of the raw score. ||v_h_unit||_E ≈ 2.81 constant across t,
# eliminating the 200× amplitude variation that caused the zero-output basin.
# See TRAINING_COLLAPSE_ANALYSIS.md for the full root-cause analysis.
EPS_PARAM = True

# ── Model (ScoreMD BBA 'large_potential' config) ──────────────────────────────
HIDDEN_DIM  = 128
NUM_LAYERS  = 3
NUM_HEADS   = 8
DIM_HEAD    = 64
INPUT_SCALE = 6.26   # BBA mean gyration radius ≈ 6.26 Å

print("Config:")
print(f"  protein      : BBA  ({N_ATOMS} Cα, manifold dim={N_ATOMS*D - D*(D+1)//2})")
print(f"  model        : GraphTransformerScoreModel  hidden={HIDDEN_DIM} layers={NUM_LAYERS} heads={NUM_HEADS} dim_head={DIM_HEAD}")
print(f"  epochs       : {N_EPOCHS}")
print(f"  batch        : {BATCH_SIZE}")
print(f"  lr           : {LR}  (cosine → 1e-5)")
print(f"  eps_param    : {EPS_PARAM}")
print(f"  data         : {LOCAL_DATA_DIR}")
print(f"  checkpoints  : {LOCAL_CKPT_DIR}")

## 5. Initialise manifold, SDE, and model

In [ ]:
# ShapeManifold constructor: dim=d, numpoints=n, alpha=delta
# (no base_point needed for training — only needed for alignment in s_geodesic/s_exp)
manifold = ShapeManifold(dim=D, numpoints=N_ATOMS, alpha=ALPHA)
sde      = ManifoldVP(manifold)

model = GraphTransformerScoreModel(
    n=N_ATOMS,
    d=D,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dim_head=DIM_HEAD,
    input_scale=INPUT_SCALE,
)

# Count parameters
rng     = jax.random.PRNGKey(SEED)
dummy_x = jnp.zeros((1, N_ATOMS * D))
dummy_t = jnp.zeros((1, 1))
params_tmp = model.init(rng, dummy_x, dummy_t)
n_params = sum(x.size for x in jax.tree_util.tree_leaves(params_tmp))
print(f"Model parameters: {n_params:,}")
# Expected: ~1,393,409

## 6. Step-time benchmark (estimates total training time)

In [ ]:
import time, optax
from training.train_manifold import make_train_step

optimizer_bm = optax.chain(
    optax.clip_by_global_norm(GRAD_CLIP),
    optax.adam(LR),
)
step_fn = make_train_step(
    model, manifold, sde, optimizer_bm,
    likelihood_weighting=False,
    eps_parameterization=EPS_PARAM,
)

rng_bm = jax.random.PRNGKey(99)
p_bm   = model.init(rng_bm, dummy_x, dummy_t)
ep_bm  = p_bm
os_bm  = optimizer_bm.init(p_bm)

B = BATCH_SIZE
xb = jax.random.normal(rng_bm, (B, N_ATOMS, D))
sb = jax.random.normal(rng_bm, (B, N_ATOMS, D))
tb = jax.random.uniform(rng_bm, (B,), minval=0.1, maxval=0.9)

# Warmup JIT
p_bm, ep_bm, os_bm, loss_bm = step_fn(p_bm, ep_bm, os_bm, xb, sb, tb)
loss_bm.block_until_ready()
print(f"JIT warmup OK  (loss={float(loss_bm):.2f})")

N_BM = 40
t0 = time.perf_counter()
acc = jnp.zeros(())
for _ in range(N_BM):
    p_bm, ep_bm, os_bm, loss_bm = step_fn(p_bm, ep_bm, os_bm, xb, sb, tb)
    acc = acc + loss_bm
acc.block_until_ready()
ms_per_step = (time.perf_counter() - t0) / N_BM * 1000

N_frames = sample['x_t'].shape[0]
steps_per_epoch = max(1, N_frames // BATCH_SIZE)
est_min = N_EPOCHS * steps_per_epoch * ms_per_step / 1000 / 60

print(f"\nStep time       : {ms_per_step:.2f} ms/step")
print(f"Steps/epoch     : {steps_per_epoch}  ({N_frames} frames / {BATCH_SIZE} batch)")
print(f"Est. total time : {est_min:.1f} min  ({N_EPOCHS} epochs)")

## 7. Training

`train_from_precomputed`:
- Preloads all repeat files into device RAM (one copy of ~600 MB total)
- Cycles repeats across epochs for stochasticity
- Accumulates loss on device, syncs only at `log_every` — no GPU→CPU stall per step
- Saves `.npz` checkpoints every `log_every` epochs to `LOCAL_CKPT_DIR`

In [ ]:
state, loss_history = train_from_precomputed(
    model=model,
    manifold=manifold,
    sde=sde,
    precomputed_dir=LOCAL_DATA_DIR,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    grad_clip=GRAD_CLIP,
    ema_decay=EMA_DECAY,
    eps_parameterization=EPS_PARAM,
    seed=SEED,
    log_every=LOG_EVERY,
    ckpt_dir=LOCAL_CKPT_DIR,
)

print("\nTraining complete.")
epochs_v, losses_v = zip(*loss_history)
print(f"  First loss : {losses_v[0]:.4f}  (epoch {epochs_v[0]})")
print(f"  Final loss : {losses_v[-1]:.4f}  (epoch {epochs_v[-1]})")

## 8. Loss curve

In [ ]:
import matplotlib.pyplot as plt

epochs_arr = np.array([e for e, _ in loss_history])
losses_arr = np.array([l for _, l in loss_history])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_arr, losses_arr, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("DSM loss (eps-param)")
ax.set_title("Phase 3.10 — BBA GraphTransformer")
ax.grid(True, alpha=0.3)
plt.tight_layout()

LOSS_PNG = "/content/310_loss_curve.png"
plt.savefig(LOSS_PNG, dpi=120)
plt.show()
print(f"Saved to {LOSS_PNG}")

## 9. Score recovery evaluation

Cosine similarity between model predictions and precomputed `s_true` targets
on the last 10% of repeat r00 (held out — not shuffled into training epochs).

With `eps_parameterization`, the model predicts `v_h_unit`; we compare
direction-only (same as training loss).

In [ ]:
data_r00 = np.load(local_files[0])
x_t_all    = data_r00["x_t"].astype(np.float32)    # (N, n, d)
s_true_all = data_r00["s_true"].astype(np.float32)  # (N, n, d)
t_all      = data_r00["t"].astype(np.float32)        # (N,)

N_held = max(512, x_t_all.shape[0] // 10)
x_heldout = jnp.array(x_t_all[-N_held:].reshape(N_held, -1))    # (N_held, nd)
s_heldout = jnp.array(s_true_all[-N_held:].reshape(N_held, -1)) # (N_held, nd)
t_heldout = jnp.array(t_all[-N_held:, None])                    # (N_held, 1)

print(f"Held-out frames : {N_held}")

# Batch inference with EMA params
score_fn = lambda x, t: model.apply(state["ema_params"], x, t)
EVAL_B = 256
preds = []
for i in range(0, N_held, EVAL_B):
    preds.append(score_fn(x_heldout[i:i+EVAL_B], t_heldout[i:i+EVAL_B]))
s_pred = jnp.concatenate(preds, axis=0)  # (N_held, nd)

def cos_sim_batch(a, b):
    a_n = a / (jnp.linalg.norm(a, axis=-1, keepdims=True) + 1e-8)
    b_n = b / (jnp.linalg.norm(b, axis=-1, keepdims=True) + 1e-8)
    return float((a_n * b_n).sum(axis=-1).mean())

cos_overall = cos_sim_batch(s_pred, s_heldout)
print(f"\ncos_sim (all t) = {cos_overall:.4f}")
print(f"  PASS" if cos_overall > 0.7 else f"  below 0.7 — check loss curve")

# Per-t-bin breakdown
t_vals = np.array(t_heldout).flatten()
print("\nPer-t-bin:")
for lo, hi in [(0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]:
    mask = (t_vals >= lo) & (t_vals < hi)
    if mask.sum() == 0:
        continue
    c = cos_sim_batch(s_pred[mask], s_heldout[mask])
    print(f"  t [{lo:.1f},{hi:.1f}]  n={mask.sum():5d}  cos_sim={c:.4f}")

## 10. Save results to Drive

In [ ]:
import shutil, json

# ── Save loss history + eval metrics as npz ───────────────────────────────────
RESULTS_NPZ = "/content/310_bba_gt_results.npz"
np.savez(
    RESULTS_NPZ,
    epochs         = np.array([e for e, _ in loss_history]),
    losses         = np.array([l for _, l in loss_history]),
    cos_sim_overall= np.array([cos_overall]),
    n_params       = np.array([n_params]),
    config_str     = np.array([str(dict(
        n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
        eps_param=EPS_PARAM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
        num_heads=NUM_HEADS, dim_head=DIM_HEAD, alpha=ALPHA,
    ))]),
)
print(f"Results npz : {RESULTS_NPZ}")

# ── Copy everything to Drive ──────────────────────────────────────────────────
files_to_copy = [
    (RESULTS_NPZ,  os.path.join(DRIVE_OUT_DIR, "310_bba_gt_results.npz")),
    (LOSS_PNG,     os.path.join(DRIVE_OUT_DIR, "310_loss_curve.png")),
]
for src_f, dst_f in files_to_copy:
    shutil.copy2(src_f, dst_f)
    print(f"Copied  {os.path.basename(src_f)}  →  Drive")

# Copy checkpoint directory to Drive
drive_ckpt_dir = os.path.join(DRIVE_OUT_DIR, "checkpoints")
if os.path.isdir(drive_ckpt_dir):
    shutil.rmtree(drive_ckpt_dir)
shutil.copytree(LOCAL_CKPT_DIR, drive_ckpt_dir)
ckpt_files = sorted(os.listdir(drive_ckpt_dir))
print(f"Checkpoints ({len(ckpt_files)} files) → Drive:checkpoints/")

print(f"\nAll results at: {DRIVE_OUT_DIR}")

## 11. Resume from checkpoint (if runtime was interrupted)

Run this cell **instead of** cell 7 if a checkpoint already exists on Drive.

In [ ]:
import re
from training.train_manifold import load_checkpoint

# Copy checkpoints from Drive back to local SSD
drive_ckpt_dir = os.path.join(DRIVE_OUT_DIR, "checkpoints")
if os.path.isdir(drive_ckpt_dir):
    if os.path.isdir(LOCAL_CKPT_DIR):
        shutil.rmtree(LOCAL_CKPT_DIR)
    shutil.copytree(drive_ckpt_dir, LOCAL_CKPT_DIR)
    print(f"Restored checkpoints from Drive → {LOCAL_CKPT_DIR}")
else:
    print(f"No checkpoint dir found at {drive_ckpt_dir}")

# Find latest epoch checkpoint
ckpts = sorted(glob.glob(os.path.join(LOCAL_CKPT_DIR, "ckpt_epoch_*.npz")))
final_ckpt = os.path.join(LOCAL_CKPT_DIR, "ckpt_final.npz")

if os.path.exists(final_ckpt):
    print("Final checkpoint found — training already complete. Load params below if needed.")
elif not ckpts:
    print("No checkpoints found — run cell 7 to start training from scratch.")
else:
    latest = ckpts[-1]
    m = re.search(r'epoch_(\d+)', os.path.basename(latest))
    done = int(m.group(1)) if m else 0
    remaining = N_EPOCHS - done
    print(f"Latest : {os.path.basename(latest)}  ({done} epochs done, {remaining} remaining)")

    example_x = np.zeros((1, N_ATOMS, D), dtype=np.float32)
    params_loaded = load_checkpoint(latest, model, manifold, sde, example_x)
    print("Params loaded OK")

    # LR continues cosine schedule from where it left off (scaled proportionally)
    lr_resume = LR * remaining / N_EPOCHS
    print(f"Resuming: {remaining} epochs, lr={lr_resume:.2e}")

    state, loss_history = train_from_precomputed(
        model=model, manifold=manifold, sde=sde,
        precomputed_dir=LOCAL_DATA_DIR,
        n_epochs=remaining, batch_size=BATCH_SIZE,
        learning_rate=lr_resume, grad_clip=GRAD_CLIP,
        ema_decay=EMA_DECAY, eps_parameterization=EPS_PARAM,
        seed=SEED, log_every=LOG_EVERY, ckpt_dir=LOCAL_CKPT_DIR,
    )

---
## Notes

### Expected results
- **Loss**: ~22 at epoch 50, decreasing to <5 by epoch 3000 with `eps_parameterization=True`
- **cos_sim held-out**: >0.9 after 3000 epochs (D16 showed 0.977 at N=128/300ep)
- **Runtime**: ~11 min on A100/H100

### If cos_sim is low (<0.7) after 3000 epochs
1. Check loss curve — still decreasing? Train longer (increase `N_EPOCHS`).
2. Check per-t-bin: if only high-t bins are low, the model learned well; high-t score is near-zero by design.
3. Verify precomputed data: regenerate with `scripts/precompute_noised_data.py --protein bba --n-repeats 10`.

### Key design choices
- **`eps_parameterization=True`**: predicts `v_h_unit` (Euclidean norm ≈ 2.81, flat across all t).
  Eliminates the 200× amplitude variation in `s_true` that caused the zero-output basin.
  See `TRAINING_COLLAPSE_ANALYSIS.md`.
- **`ALPHA=1.0`**: Both BBA and AK use δ=1.0. The ShapeManifold constructor default (0.1) is never used for actual proteins.
- **No `base_point`** in `ShapeManifold`: not needed for training (only for `s_exp`/`s_geodesic`
  alignment during inference). Precomputed targets were generated with a base point.

### Next steps after Phase 3.10
1. Reverse SDE sampling: `ManifoldEulerMaruyama` in reverse mode, 500 steps, 500 samples
2. JS divergence on pairwise Cα distances (generated vs training set)
3. Phase 4: derive the Riemannian FP residual from Kolmogorov forward equation on (M, g)